# 12. Ligand-Receptor Analysis

Ligand-receptor (LR) interaction analysis for BICRC SKNY spatial transcriptomics data.

This notebook performs:
1. Single-patient LR analysis (Pt-4 example) using squidpy and stlearn
2. CAF relabeling and integration (pCR/CR CAF vs NR CAF)
3. Per-patient LR analysis across curated signaling pathways
4. FDR correction and heatmap visualization

**Conda environment:** `skny`

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq
import stlearn as st
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from matplotlib import colors as mcolors
from matplotlib.patches import Rectangle
from scanpy.pl import MatrixPlot
from statsmodels.stats.multitest import multipletests

## Configuration

In [ ]:
DATA_DIR = "../data"
RESULTS_DIR = "../results"
FIGURES_DIR = "../figures"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
FULL_LR_LIST = [
    "SHH_PTCH1", "IHH_PTCH1",
    "TGFB1_TGFBR1", "TGFB1_TGFBR2",
    "TGFB2_TGFBR1", "TGFB2_TGFBR2",
    "TGFB3_TGFBR1", "TGFB3_TGFBR2",
    "TNF_TNFRSF1A", "TNF_TNFRSF1B",
    "IL6_IL6R", "IL6_IL6ST",
    "IL2_IL2RA", "IL2_IL2RB", "IL2_IL2RG",
    "DLL1_NOTCH1", "DLL1_NOTCH2", "DLL1_NOTCH3", "DLL1_NOTCH4",
    "DLL4_NOTCH1", "DLL4_NOTCH2", "DLL4_NOTCH3", "DLL4_NOTCH4",
    "JAG1_NOTCH1", "JAG1_NOTCH2", "JAG1_NOTCH3", "JAG1_NOTCH4",
    "WNT1_FZD1", "WNT1_FZD3", "WNT1_LRP5", "WNT1_LRP6",
    "WNT5A_FZD1", "WNT5A_FZD3", "WNT5A_LRP5", "WNT5A_LRP6",
    "FASLG_FAS", "TNFSF10_TNFRSF10A", "TNFSF10_TNFRSF10B",
    "VEGFA_KDR",
    "IFNG_IFNGR1", "IFNG_IFNGR2",
    "IFNA1_IFNAR1", "IFNA1_IFNAR2",
    "IFNB1_IFNAR1", "IFNB1_IFNAR2",
]

CURATED_LR_LIST = [
    "WNT1_FZD1", "WNT1_FZD3", "WNT1_LRP6",
    "WNT5A_FZD1", "WNT5A_FZD3",
    "TGFB1_TGFBR1", "TGFB1_TGFBR2",
    "TGFB2_TGFBR1", "TGFB2_TGFBR2",
    "TGFB3_TGFBR1", "TGFB3_TGFBR2",
    "TNF_TNFRSF1A", "TNF_TNFRSF1B",
    "TNFSF10_TNFRSF10A", "TNFSF10_TNFRSF10B",
    "DLL1_NOTCH1", "DLL1_NOTCH2", "DLL1_NOTCH3", "DLL1_NOTCH4",
    "DLL4_NOTCH1", "DLL4_NOTCH2", "DLL4_NOTCH3", "DLL4_NOTCH4",
    "JAG1_NOTCH1", "JAG1_NOTCH2", "JAG1_NOTCH3", "JAG1_NOTCH4",
    "SHH_PTCH1", "IHH_PTCH1",
    "VEGFA_KDR",
]

## Single-Patient LR Analysis (Pt-4 Example)

### Load SKNY grid

In [ ]:
grid = ad.read_h5ad(os.path.join(DATA_DIR, "BICRC_SKNY_tumor_Pt-4_Resection.h5ad"))

grid = grid[(grid.obs["euclidean"] <= 3) & (grid.obs["euclidean"] >= 0)]

grid = grid[grid.obs["subcluster"].isin([
    "Tumor", "EEF1G+ CAF", "MMP14+ CAF", "MMP11+ CAF",
    "EEF1G+ Macro", "POSTN+ Macro", "PLAUR+ Macro",
])]

grid.obs["subcluster"] = pd.Categorical(
    grid.obs["subcluster"],
    categories=[
        "EEF1G+ CAF", "MMP11+ CAF", "MMP14+ CAF",
        "EEF1G+ Macro", "PLAUR+ Macro", "POSTN+ Macro", "Tumor",
    ],
)

### Squidpy ligrec analysis

In [ ]:
res = sq.gr.ligrec(
    grid, "subcluster",
    fdr_method=None, copy=True, use_raw=False,
    threshold=0.1, seed=0, n_perms=1000, n_jobs=-1,
)

In [ ]:
sq.pl.ligrec(
    res,
    source_groups=[
        "MMP11+ CAF", "MMP14+ CAF", "EEF1G+ CAF",
        "POSTN+ Macro", "PLAUR+ Macro", "EEF1G+ Macro",
    ],
    target_groups=["Tumor"],
    swap_axes=True,
    remove_nonsig_interactions=True,
)

### stlearn CCI

In [ ]:
lrs = np.array(["POSTN_EGFR", "POSTN_PTK7", "POSTN_MAP3K7", "POSTN_ITGAV"])

st.tl.cci.run(
    grid, lrs,
    min_spots=0,
    distance=30,
    n_pairs=100,
    n_cpus=None,
)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
st.pl.lr_summary(grid, n_top=10, figsize=(2, 2), show=False, ax=ax)
plt.tight_layout()

In [ ]:
st.tl.cci.adj_pvals(
    grid, correct_axis="spot",
    pval_adj_cutoff=0.05, adj_method="fdr_bh",
)

In [ ]:
grid.obs["subcluster_"] = [i.replace("+", "") for i in grid.obs["subcluster"]]
grid.obs["subcluster_"] = pd.Categorical(grid.obs["subcluster_"])

st.tl.cci.run_cci(
    grid, "subcluster_",
    min_spots=2,
    spot_mixtures=True,
    cell_prop_cutoff=0.1,
    sig_spots=True,
    n_perms=10,
    n_cpus=None,
)

In [ ]:
st.pl.cci_check(grid, "subcluster_", figsize=(16, 5))

### Network visualization

In [ ]:
grid.uns["lr_summary"].index.values[0:5]

In [ ]:
pos_1 = st.pl.ccinet_plot(grid, "subcluster_", return_pos=True, min_counts=30)

lrs_top = grid.uns["lr_summary"].index.values[0:10]
for best_lr in lrs_top:
    st.pl.ccinet_plot(
        grid, "subcluster_", best_lr,
        min_counts=2, figsize=(10, 7.5), pos=pos_1,
    )

## CAF Relabeling & Integration

Relabel CAF subtypes:
- **CCDC80+** = pCR/CR CAF
- **EEF1G+, MMP14+, MMP11+** = NR CAF

In [ ]:
adata.obs.loc[
    adata.obs["subcluster"] == "CCDC80+ CAF", "SD_CAF_neighbor_Tumor_CAF"
] = "pCR CAF"

adata.obs["SD_CAF_neighbor_Tumor_CAF"] = (
    adata.obs["SD_CAF_neighbor_Tumor_CAF"].replace("SD CAF", "NR CAF")
)

adata.obs["CAF_annotation_"] = adata.obs["SD_CAF_neighbor_Tumor_CAF"]

In [ ]:
grid_res = grid_res[grid_res.obs["CAF_annotation_"] == "Fibroblast"]
adata = adata[adata.obs["CAF_annotation_"] != "Other"]
adata = ad.concat([adata, grid_res])

adata.obs["CAF_annotation_"].value_counts()

In [ ]:
adata_ = adata[adata.obs["CAF_annotation_"] != "Fibroblast"]
adata_caf = adata[adata.obs["CAF_annotation_"].isin(["pCR CAF", "NR CAF"])]
adata_tumor = adata[adata.obs["CAF_annotation_"].isin(["Neighbor", "Not_neighbor"])]

adata_.obs["CAF_annotation_"] = pd.Categorical(
    adata_.obs["CAF_annotation_"],
    categories=["pCR CAF", "NR CAF", "Neighbor", "Not_neighbor"],
)
adata_caf.obs["CAF_annotation_"] = pd.Categorical(
    adata_caf.obs["CAF_annotation_"],
    categories=["pCR CAF", "NR CAF"],
)
adata_tumor.obs["CAF_annotation_"] = pd.Categorical(
    adata_tumor.obs["CAF_annotation_"],
    categories=["Neighbor", "Not_neighbor"],
)

sc.tl.dendrogram(adata_, groupby="CAF_annotation_")
sc.tl.dendrogram(adata_caf, groupby="CAF_annotation_")
sc.tl.dendrogram(adata_tumor, groupby="CAF_annotation_")

In [ ]:
adata.write_h5ad(os.path.join(DATA_DIR, "caf_tumor_integrate.h5ad"))

### MatrixPlot of ligand/receptor expression

In [ ]:
ligands = {
    "SHH", "IHH", "TGFB1", "TGFB2", "TGFB3", "TNF", "IL6", "IL2",
    "VEGFA", "IFNG", "IFNA1", "IFNB1", "DLL1", "DLL4", "JAG1",
    "WNT1", "WNT5A", "FASLG", "TNFSF10",
}
receptors = {
    "PTCH1", "SMO", "TGFBR1", "TGFBR2", "TNFRSF1A", "TNFRSF1B",
    "KDR", "IFNGR1", "IFNGR2", "IFNAR1", "IFNAR2",
    "IL6R", "IL6ST", "IL2RA", "IL2RB", "IL2RG",
    "NOTCH1", "NOTCH2", "NOTCH3", "NOTCH4",
    "FZD1", "FZD3", "LRP5", "LRP6", "FAS", "TNFRSF10A", "TNFRSF10B",
}

genes_to_plot = [
    "SHH", "IHH", "PTCH1", "SMO",
    "TGFB1", "TGFB2", "TGFB3", "TGFBR1", "TGFBR2",
    "TNF", "TNFRSF1A", "TNFRSF1B",
    "IL6", "IL6R", "IL6ST",
    "IL2", "IL2RA", "IL2RB", "IL2RG",
    "DLL1", "DLL4", "JAG1", "NOTCH1", "NOTCH2", "NOTCH3", "NOTCH4",
    "WNT1", "WNT5A", "FZD1", "FZD3", "LRP5", "LRP6",
    "FASLG", "TNFSF10", "FAS", "TNFRSF10A", "TNFRSF10B",
    "VEGFA", "KDR",
    "IFNG", "IFNGR1", "IFNGR2",
    "IFNA1", "IFNB1", "IFNAR1", "IFNAR2",
]

adata.obs["CAF_annotation_"] = pd.Categorical(
    adata.obs["CAF_annotation_"].astype("category"),
    categories=["pCR CAF", "Fibroblast", "NR CAF", "Neighbor", "Not_neighbor"],
)

mp = MatrixPlot(
    adata,
    var_names=genes_to_plot,
    groupby="CAF_annotation_",
    figsize=(5, 8),
    standard_scale="var",
).swap_axes()

gene_colors = []
for gene in genes_to_plot:
    if gene in ligands:
        gene_colors.append("red")
    elif gene in receptors:
        gene_colors.append("blue")
    else:
        gene_colors.append("black")

ax_dict = mp.show(return_axes=True)

for ticklabel, color in zip(ax_dict["mainplot_ax"].get_yticklabels(), gene_colors):
    ticklabel.set_color(color)

plt.show()

## Per-Patient LR Analysis

### Loop over patients: gridding and stlearn CCI

In [ ]:
df_res = pd.DataFrame()
df_prop_res = pd.DataFrame()
df_p_res = pd.DataFrame()

for sample in adata.obs["Patient"].unique():
    adata_sample = adata[adata.obs["Patient"] == sample]
    adata_sample.uns["spatial"] = adata_sample.obsm["spatial"]

    n_ = 50
    print(f"Patient {sample}: {n_} x {n_} grid = {n_ * n_} spots")
    grid = st.tl.cci.grid(adata_sample, n_row=n_, n_col=n_, use_label="CAF_annotation_")
    print(grid.shape)

    st.tl.cci.run(
        grid, FULL_LR_LIST,
        min_spots=20,
        distance=100,
        n_pairs=100,
        n_cpus=32,
    )

    lr_info = grid.uns["lr_summary"]

    st.tl.cci.adj_pvals(
        grid, correct_axis="spot",
        pval_adj_cutoff=0.05, adj_method="fdr_bh",
    )

    st.tl.cci.run_cci(
        grid, "CAF_annotation_",
        min_spots=2,
        spot_mixtures=True,
        cell_prop_cutoff=0.1,
        sig_spots=True,
        n_perms=1,
        n_cpus=32,
    )

    pos_1 = st.pl.ccinet_plot(
        grid, "CAF_annotation_",
        return_pos=True, min_counts=30, figsize=(6, 6), font_size=15,
        title=f"All interactions ({sample})",
    )

    sig_lrs_ls = []
    sig_lrs_prop_ls = []
    sig_lrs_p_ls = []

    for lr in grid.uns["per_lr_cci_CAF_annotation_"].keys():
        cci_df = grid.uns["per_lr_cci_CAF_annotation_"][lr]
        sig_lrs_ls.append(cci_df.loc["NR CAF", "Neighbor"])
        sig_lrs_prop_ls.append(
            cci_df.loc["NR CAF", "Neighbor"] / cci_df.sum().sum()
        )
        sig_lrs_p_ls.append(
            grid.uns["per_lr_cci_pvals_CAF_annotation_"][lr].loc["NR CAF", "Neighbor"]
        )

    df = pd.DataFrame(
        sig_lrs_ls,
        index=grid.uns["per_lr_cci_CAF_annotation_"].keys(),
        columns=[sample],
    )
    df_prop = pd.DataFrame(
        sig_lrs_prop_ls,
        index=grid.uns["per_lr_cci_CAF_annotation_"].keys(),
        columns=[sample],
    )
    df_p = pd.DataFrame(
        sig_lrs_p_ls,
        index=grid.uns["per_lr_cci_pvals_CAF_annotation_"].keys(),
        columns=[sample],
    )

    df_res = pd.concat([df_res, df], axis=1)
    df_prop_res = pd.concat([df_prop_res, df_prop], axis=1)
    df_p_res = pd.concat([df_p_res, df_p], axis=1)

### Sort columns and curated LR list

In [ ]:
df_res = df_res.loc[:, sorted(df_res.columns)]
df_p_res = df_p_res.loc[:, sorted(df_p_res.columns)]

### FDR correction (Benjamini-Hochberg)

In [ ]:
p_values_flat = df_p_res.values.flatten()
_, q_values_flat, _, _ = multipletests(p_values_flat, method="fdr_bh")

df_q_res = pd.DataFrame(
    q_values_flat.reshape(df_p_res.shape),
    index=df_p_res.index,
    columns=df_p_res.columns,
)

In [ ]:
df_res_norm = df_res / (
    adata[adata.obs["CAF_annotation_"].isin(["NR CAF"])]
    .obs["Patient"].value_counts().sort_index()
)

### Heatmap visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sns.heatmap(df_res_norm.T, ax=ax, cmap="Reds")
ax.set_title("Normalized LR interaction counts")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sns.heatmap(df_res.T, ax=ax, cmap="Reds")
ax.set_title("Raw LR interaction counts")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sns.heatmap(
    df_res.loc[CURATED_LR_LIST, sorted(df_res.columns)].T,
    ax=ax, cmap="Reds",
)
ax.set_title("Curated LR interaction counts")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sns.heatmap(
    df_q_res.applymap(lambda x: np.log10(x) * -1).replace(np.inf, 10)
    .loc[CURATED_LR_LIST, sorted(df_q_res.columns)].T,
    ax=ax, cmap="Reds", vmin=0, vmax=5,
)
ax.set_title("-log10(FDR q-value) for curated LR pairs")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df_res.T, ax=ax, cmap="Reds", cbar=True)

for y in range(df_q_res.T.shape[0]):
    for x in range(df_q_res.T.shape[1]):
        if df_p_res.T.iloc[y, x] < 0.001:
            rect = Rectangle((x, y), 1, 1, fill=False, edgecolor="black", linewidth=2)
            ax.add_patch(rect)

ax.set_title("LR interaction counts (black border = p < 0.001)")
plt.tight_layout()
plt.show()

## Results Export

In [ ]:
df_res.to_csv(os.path.join(RESULTS_DIR, "lr_each_patients.txt"), sep="\t")
df_p_res.to_csv(os.path.join(RESULTS_DIR, "lr_pvalues_each_patients.txt"), sep="\t")
df_q_res.to_csv(os.path.join(RESULTS_DIR, "lr_qvalues_each_patients.txt"), sep="\t")

print("Results exported to", RESULTS_DIR)